In [ ]:
# Install dependencies (only needed once)
!pip install reportlab pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.0 MB/s eta 0:00:00


In [ ]:
# Import library
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.pdfgen import canvas
import reportlab.lib.utils as utils
import pandas as pd
import os, re, glob
import os
from datetime import datetime
from zipfile import ZipFile
from IPython.display import FileLink
from IPython.display import IFrame

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Step 3: Set path to your CSV file in Drive
# 🔹 Change this path to the actual location of your file in Drive
csv_path = '/content/drive/MyDrive/automation_report_sample/deforestation_cases_sample.csv' # folder where IMG_3744_*.jpg are stored
output_dir = '/content/drive/MyDrive/automation_report_sample/deforestation_pdfs'
logo_path  = '/content/drive/MyDrive/automation_report_sample/logo/ccdi_logo.png'

In [ ]:
# Define Photo paths
photo_paths = [
    '/content/drive/MyDrive/automation_report_sample/photos/IMG_3744_1.jpg',
    '/content/drive/MyDrive/automation_report_sample/photos/IMG_3744_2.jpg',
    '/content/drive/MyDrive/automation_report_sample/photos/IMG_3744_3.jpg'
]

In [ ]:
# Step 4: Load CSV
df = pd.read_csv(csv_path)
print(f"✅ Loaded {len(df)} rows from {csv_path}")

✅ Loaded 20 rows from /content/drive/MyDrive/automation_report_sample/deforestation_cases_sample.csv


In [ ]:
# ---------- 5) Helpers ----------

def get_image(path, width=None, height=None):
    """
    Load an image and optionally scale by target width or height.
    Returns (img, scaled_width, scaled_height) or (None, 0, 0) if it fails.
    """
    try:
        img = utils.ImageReader(path)
        iw, ih = img.getSize()

        if width and not height:
            ratio = width / float(iw)
            return img, width, ih * ratio

        if height and not width:
            ratio = height / float(ih)
            return img, iw * ratio, height

        return img, iw, ih

    except Exception as e:
        print(f"[get_image] Could not load image {path}: {e}")
        return None, 0, 0


def draw_wrapped_text(c, text, x, y, max_width, leading=14,
                      font="Helvetica", size=10):
    """
    Draw text that wraps to the next line if it exceeds max_width.
    Returns the updated y position.
    """
    c.setFont(font, size)
    words = str(text).split()
    line = ""

    for w in words:
        test_line = (line + " " + w).strip()
        if c.stringWidth(test_line, font, size) <= max_width:
            line = test_line
        else:
            c.drawString(x, y, line)
            y -= leading
            line = w

    if line:
        c.drawString(x, y, line)
        y -= leading

    return y


def safe_value(val, default="N/A"):
    """Convert NaN or None to a nice default string."""
    if pd.isna(val):
        return default
    return val


def field_block(c, label, value, x, y, label_w, page_w,
                leading=14, font="Helvetica", size=10):
    """
    Draw a 'Label: value' block with wrapping for the value.
    Handles NaN by printing 'N/A' instead.
    Returns the updated y position.
    """
    c.setFont("Helvetica-Bold", 10)
    c.drawString(x, y, f"{label}:")
    c.setFont(font, size)

    start_x = x + label_w + 6
    max_width = page_w - start_x - 2 * cm

    clean_val = safe_value(value, default="N/A")
    return draw_wrapped_text(c, clean_val, start_x, y, max_width, leading=leading)


def sanitize(s):
    """Make a string safe for filenames."""
    s = str(s)
    s = s.replace("(", "_").replace(")", "_").replace(" ", "_")
    s = re.sub(r"__+", "_", s)
    return s


# ---------- 6) Photo configuration (two images below info) ----------

# Define which two photos you want for each row (3 example rows here)
photo_pairs = [
    [
        "/content/drive/MyDrive/automation_report_sample/photos/IMG_3744_1.jpg",
        "/content/drive/MyDrive/automation_report_sample/photos/IMG_3744_2.jpg",
    ],
    [
        "/content/drive/MyDrive/automation_report_sample/photos/IMG_3744_2.jpg",
        "/content/drive/MyDrive/automation_report_sample/photos/IMG_3744_3.jpg",
    ],
    [
        "/content/drive/MyDrive/automation_report_sample/photos/IMG_3744_1.jpg",
        "/content/drive/MyDrive/automation_report_sample/photos/IMG_3744_3.jpg",
    ],
]


def get_photos_for_row(row, idx):
    """
    Return a list of photos for this row.
    Here: we use pre-defined pairs based on row index.
    """
    if 0 <= idx < len(photo_pairs):
        return photo_pairs[idx]
    return []


def draw_photo_row(c, photos, x, y, page_w, margin):
    """
    Draw up to 3 photos directly below the information block.
    Typically: 2 landscape photos side-by-side (like your example).
    Returns the updated y position.
    """
    if not photos:
        return y

    # Label above photos
    c.setFont("Helvetica-Bold", 10)
    c.drawString(x, y, "Photo Evidence:")
    y -= 12

    max_h = 7 * cm   # max photo height

    cols = min(len(photos), 3)
    gap = 0.6 * cm
    total_gap = gap * (cols - 1)
    available_w = (page_w - 2 * margin - total_gap) / cols

    max_height_row = 0.0
    for i, p in enumerate(photos[:cols]):
        img, iw, ih = get_image(p, width=available_w)
        if not img:
            continue

        scale = min(1.0, max_h / ih) if ih else 1.0
        w, h = iw * scale, ih * scale

        x_pos = x + i * (available_w + gap)
        y_pos = y - h
        c.drawImage(
            img,
            x_pos, y_pos,
            width=w, height=h,
            preserveAspectRatio=True,
            mask='auto'
        )
        max_height_row = max(max_height_row, h)

    return y - max_height_row - 10


# ---------- 7) MAIN PDF GENERATOR (header + logo + info + photos) ----------

def export_case_pdfs_from_csv(df, outdir=output_dir):
    """
    Generate one PDF per row in df.
    - Uses global logo_path and photo_pairs.
    - Returns list of created file paths.
    """
    os.makedirs(outdir, exist_ok=True)

    # skip any CSV columns that are photo/picture references
    cols_to_render = [
        col for col in df.columns
        if "photo" not in col.lower() and "picture" not in col.lower()
    ]

    exported_files = []

    for idx, row in df.iterrows():
        case_val = row.get("Defo Case Number", f"Row_{idx + 1}")
        case_no = sanitize(case_val)
        filepath = os.path.join(outdir, f"Deforestation_Case_{case_no}.pdf")

        c = canvas.Canvas(filepath, pagesize=A4)
        width, height = A4
        margin = 1.7 * cm

        # ----- HEADER + LOGO -----
        header_y = height - margin

        # Title
        c.setFont("Helvetica-Bold", 16)
        c.drawString(margin, header_y, "CARAGA CARBON DEVELOPMENT, INC.")

        # Subtitle
        c.setFont("Helvetica", 11)
        c.drawString(margin, header_y - 18, "Deforestation Alert Validation — Field Report")

        # Logo on top-right aligned with title
        if 'logo_path' in globals() and os.path.exists(logo_path):
            logo_img, lw, lh = get_image(logo_path, width=4.0 * cm)
            if logo_img:
                logo_x = width - lw - margin
                # tweak the 0.65 factor slightly if you want the logo higher/lower
                logo_y = header_y - (lh * 0.50)
                c.drawImage(
                    logo_img,
                    logo_x, logo_y,
                    width=lw,
                    height=lh,
                    mask='auto'
                )

        # Horizontal line
        line_y = header_y - 30
        c.line(margin, line_y, width - margin, line_y)

        # Start body area
        x = margin
        y = line_y - 20

        # ----- TEXT FIELDS (info block) -----
        # ----- TEXT FIELDS (info block) -----
        label_width = 170
        for col in cols_to_render:
            y = field_block(c, col, row[col], x, y, label_width, width)
            y -= 4

        # ----- PHOTOS DIRECTLY BELOW INFO -----
        photos = get_photos_for_row(row, idx)
        if photos:
            y -= 8
            y = draw_photo_row(c, photos, x, y, width, margin)

        # ----- FOOTER -----
        c.setFont("Helvetica-Oblique", 8)
        c.drawString(
            margin,
            margin,
            f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}"
        )

        c.showPage()
        c.save()
        exported_files.append(filepath)

    print(f"✅ PDFs created in: {outdir}")
    return exported_files

In [ ]:
exported = export_case_pdfs_from_csv(df.head(1), outdir=output_dir)
exported[:3]

✅ PDFs created in: /content/drive/MyDrive/automation_report_sample/deforestation_pdfs


['/content/drive/MyDrive/automation_report_sample/deforestation_pdfs/Deforestation_Case_1_E-12_.pdf']